In [7]:
from notebooks._utils import calculate_parallel_ensemble_accuracy
from notebooks._utils import calculate_baseline_accuracy, get_layers, calculate_oracle_accuracy


num_fewshots = 0
dataset_root = "/home/xzhao/workspace/GYB_self-ensemble/datasets"
all_models = ["llama3.2_3b", "qwen2.5_3b", "qwen3_4b", "phi3.5_mini", "qwen3_30b", "gpt_20b"]

In [8]:
ds_names = ["myriadlama", "hotpot"]

for ds_name in ds_names:
    print(f"\n################### Dataset: {ds_name} ###################")
    for model_name in all_models:
        print(f"\n=================== Model: {model_name} ===================")
        dump_file_prefix = f"{dataset_root}/{ds_name}/{model_name}/{ds_name}."
        print("---- Calculating baseline ----")
        calculate_baseline_accuracy(dataset_root, ds_name, model_name, num_fewshots, is_multichoice=False, by_probs=False)
        
        print("---- Calculating Oracle ----")
        calculate_oracle_accuracy(dataset_root, ds_name, model_name, num_fewshots, is_multichoice=False, by_probs=False)
        
        print("---- Logits Ensemble + Layer Average ----")
        calculate_parallel_ensemble_accuracy(dump_file_prefix, repeat_paras=False,
            num_paraphrases=5, num_fewshots=num_fewshots, num_samples=5 if ds_name=="myriadlama" else 1,
            logits_ensemble_method="avg",
            ensemble_method="layer_output_avg",
            multilayer=True, ensemble_alpha=1.0, 
            ensemble_layer=get_layers(model_name), token_mode="last", 
            by_probs=False, is_multichoice=False)
    


################### Dataset: myriadlama ###################

=================== Model: llama3.2_3b ===================
---- Calculating baseline ----
Acc: 0.3417 ==> 🏷️ baseline of average accuracy per-paraphrase
---- Calculating Oracle ----
Acc: 0.6815 ==> 🏷️ Oracle accuracy per-paraphrase
---- Logits Ensemble + Layer Average ----
Acc: 0.4553 ==> 🏷️ 5paras 0shots  layer_output_avg layer21 Multilayer alpha1.0 token-last

=================== Model: qwen2.5_3b ===================
---- Calculating baseline ----
Acc: 0.1962 ==> 🏷️ baseline of average accuracy per-paraphrase
---- Calculating Oracle ----
Acc: 0.5540 ==> 🏷️ Oracle accuracy per-paraphrase
---- Logits Ensemble + Layer Average ----
Acc: 0.3160 ==> 🏷️ 5paras 0shots  layer_output_avg layer27 Multilayer alpha1.0 token-last

=================== Model: qwen3_4b ===================
---- Calculating baseline ----
Acc: 0.2413 ==> 🏷️ baseline of average accuracy per-paraphrase
---- Calculating Oracle ----
Acc: 0.5560 ==> 🏷️ Oracle accu

In [9]:

for ds_name in ["commonsense", "mmlu", "logiqa"]:
# for ds_name in ["commonsense"]:
# for ds_name in ["logiqa"]:
    print(f"\n################### Dataset: {ds_name} ###################")

    for model_name in all_models:
        print(f"\n---------- Model: {model_name} ----------")
        dump_file_prefix = f"{dataset_root}/{ds_name}/{model_name}/{ds_name}paraphrase."
        print("---- 📜 GENERATION: Calculating baseline ----")
        calculate_baseline_accuracy(dataset_root, ds_name, model_name, num_fewshots, by_probs=False, is_multichoice=True)
        
        print("---- 📜 GENERATION: Calculating Oracle ----")
        calculate_oracle_accuracy(dataset_root, ds_name, model_name, num_fewshots, by_probs=False, is_multichoice=True)

        print("---- 📜 GENERATION: Logits Ensemble + Layer Average ----")
        calculate_parallel_ensemble_accuracy(dump_file_prefix, repeat_paras=False,
            num_paraphrases=5, num_fewshots=num_fewshots, num_samples=1,
            logits_ensemble_method="avg",
            ensemble_method="layer_output_avg",
            multilayer=True, ensemble_alpha=1.0, 
            ensemble_layer=get_layers(model_name), token_mode="last", 
            use_generation=True, by_probs=False, is_multichoice=True)
        
        
        print("\n---- 🎲 PROBABILITY: Calculating baseline ----")
        calculate_baseline_accuracy(dataset_root, ds_name, model_name, num_fewshots, by_probs=True)
        
        print("---- 🎲 PROBABILITY: Calculating Oracle ----")
        calculate_oracle_accuracy(dataset_root, ds_name, model_name, num_fewshots, by_probs=True)
    
        print("---- 🎲 PROBABILITY: Logits Ensemble + Layer Average ----")
        calculate_parallel_ensemble_accuracy(dump_file_prefix, repeat_paras=False,
            num_paraphrases=5, num_fewshots=num_fewshots, num_samples=1,
            logits_ensemble_method="avg",
            ensemble_method="layer_output_avg",
            multilayer=True, ensemble_alpha=1.0, 
            ensemble_layer=get_layers(model_name), token_mode="last", 
            use_generation=True, by_probs=True, is_multichoice=True)
    


################### Dataset: commonsense ###################

---------- Model: llama3.2_3b ----------
---- 📜 GENERATION: Calculating baseline ----
Acc: 0.2490 ==> 🏷️ baseline of average accuracy per-paraphrase
---- 📜 GENERATION: Calculating Oracle ----
Acc: 0.3660 ==> 🏷️ Oracle accuracy per-paraphrase
---- 📜 GENERATION: Logits Ensemble + Layer Average ----
Acc: 0.5130 ==> 🏷️ 5paras 0shots  layer_output_avg layer21 Multilayer alpha1.0 token-last

---- 🎲 PROBABILITY: Calculating baseline ----
Acc: 0.2496 ==> 🏷️ baseline of average accuracy per-paraphrase
---- 🎲 PROBABILITY: Calculating Oracle ----
Acc: 0.3720 ==> 🏷️ Oracle accuracy per-paraphrase
---- 🎲 PROBABILITY: Logits Ensemble + Layer Average ----
Acc: 0.5130 ==> 🏷️ 5paras 0shots  layer_output_avg layer21 Multilayer alpha1.0 token-last

---------- Model: qwen2.5_3b ----------
---- 📜 GENERATION: Calculating baseline ----
Acc: 0.2558 ==> 🏷️ baseline of average accuracy per-paraphrase
---- 📜 GENERATION: Calculating Oracle ----
Acc: 0

In [10]:
import pandas as pd

df = pd.read_feather("/home/xzhao/workspace/GYB_self-ensemble/datasets/mmlu/gpt_20b/baseline_per_prompt.0shots.feather")

In [11]:
df

,uuid,answers,paraphrase,prompt,prediction,generation,is_orig,choices_label,choices_text,answer_label,labels,label_probs,predict_lemma,generation_lemmas,answer_lemmas
0,000d45b3136678e85a763c55a797fe44,C,An object moving along a line has velocity v(t...,You are given a multiple-choice question.\nCho...,We need to? The problem? The problem: C.We nee...,We need to? The problem? The problem: C.We nee...,True,"[A, B, C, D]","[none, one, two, three]",C,"[ A, B, C, D]","[5.9371814131736755e-08, 6.664777174592018e-09...","[we, need, to, ?, the, problem, ?, the, proble...","[we, need, to, ?, the, problem, ?, the, proble...",[[c]]
1,002fba5ca35de3b4c7398c42e18b04e2,D,An externality,You are given a multiple-choice question.\nCho...,We need? The question? The question? The quest...,We need? The question? The question? The quest...,True,"[A, B, C, D]",[causes the equilibrium price to be artificial...,D,"[ A, B, C, D]","[1.2814998626708984e-05, 4.330649971961975e-08...","[we, need, ?, the, question, ?, the, question,...","[we, need, ?, the, question, ?, the, question,...",[[d]]
2,00329ac869f57b21c5e3d19f4d96ddcc,A,What value of y makes y + 2.9 = 11 true?,You are given a multiple-choice question.\nCho...,We need to correct?,We need to correct?\n\nWe need y + ..<one 3\n\...,True,"[A, B, C, D]","[8.1, 8.9, 9.1, 13.9]",A,"[ A, B, C, D]","[6.4373016357421875e-06, 1.0505318641662598e-0...","[we, need, to, correct, ?]","[we, need, to, correct, ?, \n\n, we, need, y, +]",[[a]]
3,00d52dfe2593a45237af1f829b3c6ce6,B,Emma is browsing a popular website when she ru...,You are given a multiple-choice question.\nCho...,We need?,We need?\n\nWe need? The question? The questio...,True,"[A, B, C, D]","[proactive, reactive, manipulative, manipulative]",B,"[ A, B, C, D]","[3.5390257835388184e-07, 2.9685907065868378e-0...","[we, need, ?]","[we, need, ?, \n\n, we, need, ?, the, question...",[[b]]
4,00eed3612ce142ac2617cc75e6e77747,B,Which of the following is not an observation o...,You are given a multiple-choice question.\nCho...,We need?,We need?\n\nWe need to do we need to? The prob...,True,"[A, B, C, D]",[There is heritable variation among individual...,B,"[ A, B, C, D]","[3.2335519790649414e-06, 1.0652001947164536e-0...","[we, need, ?]","[we, need, ?, \n\n, we, need, to, do, we, need...",[[b]]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,fef51106e56f3fd800ed1c5378596bf8,C,What condition produces elevated pressure with...,You are given a multiple-choice question.\nCho...,We need? The Squee? The Squee,We need? The Squee? The Squee\n\nAnswer =? The...,False,"[A, B, C, D]","[astigmatism, cataract, glaucoma, retinitis]",C,"[ A, B, C, D]","[2.9616057872772217e-07, 3.1868694350123405e-0...","[we, need, ?, the, squee, ?, the, squee]","[we, need, ?, the, squee, ?, the, squee, \n\n,...",[[c]]
4996,fefadafdbefc65077b5be7e6ae2c5386,C,Which change to the experimental geometry or l...,You are given a multiple-choice question.\nCho...,We need?,We need? \nWe have to: The\nWe have no?**? \nW...,False,"[A, B, C, D]","[Use light of a shorter wavelength., Move the ...",C,"[ A, B, C, D]","[7.674098014831543e-07, 6.51925802230835e-08, ...","[we, need, ?]","[we, need, ?, \n, we, have, to, :, the, \n, we...",[[c]]
4997,ff593fe4da18e7e37602334a2925492a,C,What true assertion applies to the many method...,You are given a multiple-choice question.\nCho...,We need?,We need? \nWe need to ..?? \nWe need to? The S...,False,"[A, B, C, D]",[community-based (halfway-house) treatments ha...,C,"[ A, B, C, D]","[1.4454126358032227e-06, 2.0605511963367462e-0...","[we, need, ?]","[we, need, ?, \n, we, need, to]",[[c]]
4998,ffcadb4334eae438177df628ec1b67a5,D,"When the ambient temperature increases, which ...",You are given a multiple-choice question.\nCho...,We need?,We need? \nWe have to\n\nWe have to\n\nWe have...,False,"[A, B, C, D]","[decreasing salt retention, increasing respira...",D,"[ A, B, C, D]","[5.438923835754395e-07, 1.594889909029007e-08,...","[we, need, ?]","[w